In [1]:
!pip install tensorflow matplotlib opencv-python

In [2]:
import tensorflow as tf
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

In [3]:
import os

os.makedirs("dataset/defect", exist_ok=True)
os.makedirs("dataset/good", exist_ok=True)

In [4]:
for i in range(50):
    img = np.ones((128,128,3)) * 255
    cv2.circle(img, (np.random.randint(20,100), np.random.randint(20,100)),
               10, (0,0,0), -1)
    cv2.imwrite(f"dataset/defect/defect_{i}.png", img)

In [5]:
for i in range(50):
    img = np.ones((128,128,3)) * 255
    cv2.imwrite(f"dataset/good/good_{i}.png", img)

In [6]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    "dataset",
    target_size=(128,128),
    batch_size=16,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    "dataset",
    target_size=(128,128),
    batch_size=16,
    class_mode='binary',
    subset='validation'
)

Found 80 images belonging to 2 classes.
Found 20 images belonging to 2 classes.


In [7]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
history = model.fit(train_data, epochs=5, validation_data=val_data)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 665ms/step - accuracy: 0.4922 - loss: 4.0145 - val_accuracy: 0.5000 - val_loss: 1.5048
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 400ms/step - accuracy: 0.5278 - loss: 0.9993 - val_accuracy: 0.5000 - val_loss: 0.6917
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 398ms/step - accuracy: 0.4809 - loss: 0.6978 - val_accuracy: 0.5000 - val_loss: 0.6743
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 415ms/step - accuracy: 0.7253 - loss: 0.6573 - val_accuracy: 1.0000 - val_loss: 0.6141
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 2s 404ms/step - accuracy: 0.7903 - loss: 0.6382 - val_accuracy: 0.7500 - val_loss: 0.6102


In [9]:
def severity_score(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    dark_pixels = np.sum(gray < 50)
    total_pixels = gray.size
    return (dark_pixels / total_pixels) * 100

In [10]:
def inspect_image(image_path):
    img = cv2.imread(image_path)
    resized = cv2.resize(img, (128,128)) / 255.0
    prediction = model.predict(np.expand_dims(resized, axis=0))[0][0]

    severity = severity_score(image_path)

    if prediction > 0.5:
        status = "DEFECT DETECTED"
        comment = "Structural irregularity identified. Inspection recommended."
    else:
        status = "NO DEFECT"
        comment = "Surface appears uniform and acceptable."

    return status, prediction, severity, comment

In [18]:
status, conf, sev, comment = inspect_image("dataset/defect/defect_1.png")

print("Inspection Result:", status)
print("Confidence Score:", round(conf,2))
print("Severity Level (%):", round(sev,2))
print("AI Remark:", comment)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step
Inspection Result: NO DEFECT
Confidence Score: 0.39
Severity Level (%): 1.93
AI Remark: Surface appears uniform and acceptable.
